In [ ]:
import sys

def get_rotations(text):
    """Generate all rotations of a string."""
    return [text[i:] + text[:i] for i in range(len(text))]

class FMIndex:
    def __init__(self, text):
        print("Starting FM-Index construction...")
        if '$' not in text:
            text += '$'

        # 1. Get BWT
        print("  1. Generating rotations and Suffix Array...")
        rotations = sorted(get_rotations(text))
        self.bwt = "".join([r[-1] for r in rotations])
        print(f"  BWT created (length {len(self.bwt)}).")

        # 2. Build C Table
        print("  2. Building Counts Table (C)...")
        self.counts = dict()
        for char in self.bwt:
            self.counts[char] = self.counts.get(char, 0) + 1

        self.c_table = dict()
        total = 0
        for char in sorted(self.counts.keys()):
            self.c_table[char] = total
            total += self.counts[char]
        print("  C Table created.")

        # 3. Build (full, unoptimized) Occ Table
        print("  3. Building Occurrence Table (Occ)...")
        self.occ_table = []
        running_counts = {char: 0 for char in self.counts}
        for char in self.bwt:
            self.occ_table.append(running_counts.copy())
            running_counts[char] += 1
        print("  Occ Table created.")
        print("FM-Index construction complete.")

    def _get_occ(self, char, rank):
        """Get occurrences of char before a given rank."""
        if char not in self.counts or rank < 0:
            return 0
        if rank >= len(self.occ_table):
            rank = len(self.occ_table) -1
        return self.occ_table[rank].get(char, 0)

    def search(self, pattern):
        """
        Performs backward search to find the range of a pattern.
        This version only counts the matches.
        """
        if not pattern or pattern[0] not in self.c_table:
            return 0

        # Initialize range with the last character
        last_char = pattern[-1]
        start = self.c_table[last_char]
        end = self.c_table[last_char] + self.counts[last_char]

        # Iterate backwards over the rest of the pattern
        for char in reversed(pattern[:-1]):
            if start >= end:
                return 0 # No matches

            # This is the core LF-Mapping step
            start = self.c_table[char] + self._get_occ(char, start)
            end = self.c_table[char] + self._get_occ(char, end)

        return end - start

# --- USAGE EXAMPLE ---
if __name__ == '__main__':
    # Using a larger, more realistic string
    genome = "ATGATGCATATGCATGATGCATATGCATGATGCATATGCATGATGCATATGCATGATGCATATGCATGATGCAT"
    print(f"Building index for a genome of length {len(genome)}...")

    fm = FMIndex(genome)

    print("\n--- Searching ---")

    short_pattern1 = "ATG"
    count1 = fm.search(short_pattern1)
    print(f"The short pattern '{short_pattern1}' was found {count1} times.")

    short_pattern2 = "TCA"
    count2 = fm.search(short_pattern2)
    print(f"The short pattern '{short_pattern2}' was found {count2} times.")

    long_pattern = "CATATGCATG"
    count3 = fm.search(long_pattern)
    print(f"The long pattern '{long_pattern}' was found {count3} times.")

    nonexistent_pattern = "XYZ"
    count4 = fm.search(nonexistent_pattern)
    print(f"The non-existent pattern '{nonexistent_pattern}' was found {count4} times.")

Building index for a genome of length 74...
Starting FM-Index construction...
  1. Generating rotations and Suffix Array...
  BWT created (length 75).
  2. Building Counts Table (C)...
  C Table created.
  3. Building Occurrence Table (Occ)...
  Occ Table created.
FM-Index construction complete.

--- Searching ---
The short pattern 'ATG' was found 16 times.
The short pattern 'TCA' was found 0 times.
The long pattern 'CATATGCATG' was found 4 times.
The non-existent pattern 'XYZ' was found 0 times.


In [3]:
pip install pydivsufsort

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import sys
from pydivsufsort import divsufsort

class GenomeFMIndex:
    def __init__(self, text, sa_rate=32, c_interval=128):
        """
        Builds a memory-efficient FM-Index for a long genome string.

        Args:
            text (str): The genome sequence.
            sa_rate (int): Suffix Array sampling rate. Higher saves more memory.
            c_interval (int): Occurrence table checkpointing interval.
        """
        print("Starting Production FM-Index construction...")
        if not text.endswith('$'):
            text += '$'
        n = len(text)
        self.n = n
        self.sa_rate = sa_rate
        self.c_interval = c_interval

        # 1. Build Suffix Array and then BWT
        print(f"  1. Building Suffix Array for text of length {n}...")
        sa = divsufsort(text)
        print("  Suffix Array built. Generating BWT...")
        # This is a fast way to build BWT from SA
        self.bwt = "".join([text[sa[i] - 1] for i in range(n)])
        print("  BWT created.")

        # 2. Build C Table (same as before)
        print("  2. Building Counts Table (C)...")
        self.counts = {char: self.bwt.count(char) for char in set(self.bwt)}
        self.c_table = {}
        total = 0
        for char in sorted(self.counts.keys()):
            self.c_table[char] = total
            total += self.counts[char]
        print("  C Table created.")

        # 3. Build Checkpointed Occurrence Table
        print(f"  3. Building Checkpointed Occurrence Table (Interval: {c_interval})...")
        self.occ_checkpoints = []
        running_counts = {char: 0 for char in self.counts}
        for i, char in enumerate(self.bwt):
            if i % c_interval == 0:
                self.occ_checkpoints.append(running_counts.copy())
            running_counts[char] += 1
        print("  Occ Table created.")

        # 4. Build Sampled Suffix Array
        print(f"  4. Building Sampled Suffix Array (Rate: {sa_rate})...")
        self.sampled_sa = {i: sa[i] for i in range(n) if i % sa_rate == 0}
        print("  Sampled SA created.")
        print("FM-Index construction complete.")

    def _get_occ(self, char, rank):
        """Get occurrences of char before a given rank using checkpoints."""
        if char not in self.counts or rank <= 0:
            return 0

        # Find the nearest checkpoint
        checkpoint_idx = rank // self.c_interval
        start_pos = checkpoint_idx * self.c_interval

        # Get count from checkpoint and scan the rest of the way
        count = self.occ_checkpoints[checkpoint_idx].get(char, 0)
        count += self.bwt[start_pos:rank].count(char)

        return count

    def _lf_map(self, rank):
        """Performs one step of LF-mapping."""
        char = self.bwt[rank]
        return self.c_table[char] + self._get_occ(char, rank)

    def _get_pos(self, rank):
        """Finds the original text position for a given BWT rank using the sampled SA."""
        steps = 0
        while rank not in self.sampled_sa:
            rank = self._lf_map(rank)
            steps += 1
        return self.sampled_sa[rank] + steps

    def find_matches(self, pattern):
        """
        Finds the start positions of all exact occurrences of a pattern.
        """
        if not pattern:
            return []

        # --- Backward Search ---
        last_char = pattern[-1]
        if last_char not in self.c_table:
            return []

        start = self.c_table[last_char]
        end = self.c_table[last_char] + self.counts[last_char]

        for char in reversed(pattern[:-1]):
            if start >= end:
                return []
            start = self.c_table[char] + self._get_occ(char, start)
            end = self.c_table[char] + self._get_occ(char, end)

        if start >= end:
            return []

        # --- Resolve Positions ---
        return [self._get_pos(i) for i in range(start, end)]

# --- USAGE EXAMPLE ---
if __name__ == '__main__':
    # Simulate loading a "long" genome sequence
    # For a real case: with open('genome.fa') as f: genome = f.read()...
    base_seq = "ATGATGCATATGC" * 1000 # Make it longer to show the system works
    genome = base_seq + "TCACTACTCTCA" + base_seq
    print(f"Building index for a genome of length {len(genome)}...")

    # Build the index
    fm = GenomeFMIndex(genome, sa_rate=16, c_interval=64)

    print("\n--- Searching ---")

    # This pattern is very short and appears many times
    short_pattern = "ATG"
    positions1 = fm.find_matches(short_pattern)
    print(f"Found '{short_pattern}' at {len(positions1)} locations. First few: {positions1[:5]}")

    # This pattern is unique and short
    unique_short_pattern = "TCACTACTCTCA"
    positions2 = fm.find_matches(unique_short_pattern)
    print(f"Found '{unique_short_pattern}' at {len(positions2)} locations: {positions2}")

    # This pattern does not exist
    nonexistent_pattern = "XYZ"
    positions3 = fm.find_matches(nonexistent_pattern)
    print(f"Found '{nonexistent_pattern}' at {len(positions3)} locations: {positions3}")

Building index for a genome of length 26012...
Starting Production FM-Index construction...
  1. Building Suffix Array for text of length 26013...
  Suffix Array built. Generating BWT...
  BWT created.
  2. Building Counts Table (C)...
  C Table created.
  3. Building Checkpointed Occurrence Table (Interval: 64)...
  Occ Table created.
  4. Building Sampled Suffix Array (Rate: 16)...
  Sampled SA created.
FM-Index construction complete.

--- Searching ---


C:\Users\Ana\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\pydivsufsort\divsufsort.py:103: UserWarning: converting str argument uses more memory
  inp_p = _get_bytes_pointer(inp)


Found 'ATG' at 6000 locations. First few: [25999, 25986, 25973, 25960, 25947]
Found 'TCACTACTCTCA' at 1 locations: [13000]
Found 'XYZ' at 0 locations: []


In [3]:
with open('sequence.fasta', 'r') as f:
    lines = f.readlines()
    genome_string = ''.join(line.strip() for line in lines if not line.startswith('>'))
fm = GenomeFMIndex(genome_string)

patterns = ["ATGATG", "CTCTCTA", "TCACTACTCTCA"]
for pattern in patterns:
    positions = fm.find_matches(pattern)
    print(len(positions))

Starting Production FM-Index construction...
  1. Building Suffix Array for text of length 148379592...
  Suffix Array built. Generating BWT...
  Suffix Array built. Generating BWT...
  BWT created.
  2. Building Counts Table (C)...
  BWT created.
  2. Building Counts Table (C)...
  C Table created.
  3. Building Checkpointed Occurrence Table (Interval: 128)...
  C Table created.
  3. Building Checkpointed Occurrence Table (Interval: 128)...
  Occ Table created.
  4. Building Sampled Suffix Array (Rate: 32)...
  Occ Table created.
  4. Building Sampled Suffix Array (Rate: 32)...
  Sampled SA created.
FM-Index construction complete.
  Sampled SA created.
FM-Index construction complete.
43127
43127
10219
8
10219
8
